In [ ]:
import pandas as pd

# Load file
df = pd.read_csv("../data/ner/ner_extract_entity_dataset.csv")

def convert_to_bio(row):
    text = row["text"]
    tokens = text.split()
    labels = ["O"] * len(tokens)

    # -------- P2P --------
    if row["intent"] == "p2p_transfer":
        
        amount = str(row.get("amount", "")).strip()
        currency = str(row.get("currency", "")).strip()
        recipient = str(row.get("recipient", "")).strip()

        for i, token in enumerate(tokens):
            
            if token == amount:
                labels[i] = "B-AMOUNT"
            
            elif token == currency:
                labels[i] = "B-CURRENCY"
            
            elif recipient and recipient in token:
                labels[i] = "B-RECIPIENT"

    # -------- PAY BILL --------
    elif row["intent"] == "pay_bill":
        
        bill = str(row.get("bill_type", "")).strip()
        
        if bill:
            bill_tokens = bill.split()
            for i in range(len(tokens)):
                if tokens[i:i+len(bill_tokens)] == bill_tokens:
                    labels[i] = "B-BILL_TYPE"
                    for j in range(1, len(bill_tokens)):
                        labels[i+j] = "I-BILL_TYPE"

    return tokens, labels


bio_data = df.apply(convert_to_bio, axis=1)

df["tokens"] = bio_data.apply(lambda x: x[0])
df["ner_tags"] = bio_data.apply(lambda x: x[1])

df.to_csv("../data/ner/BIO_tagging_dataset.csv", index=False)

print("✅ BIO dataset saved.")

✅ BIO dataset saved.
